# ML-07 — Baseline Action Score and Top-20 Review

Lane: **Refresh / Content Opportunity Scoring** (Lane 2).
This notebook builds the transparent rule baseline every Week-5 model must beat.

**Structure:**
1. Two signal bucket-table checks (at least one flag-linked) with verdicts
2. Rule encoding: score, reason code, action label → ranked CSV
3. Top-10 hand review with "what would make it wrong"
4. Weak picks + leakage check
5. Self-check

> Skill router: loaded `building-baselines` + `flyrank/flyrank-data` (per `skills/README.md`).

## 0. Setup — load the starter data

The 30k-row starter CSV lives at `data/raw/content_refresh_anonymized.csv`. Rate columns are
×100 percentages (`ctr = 0.76` means 0.76%). `avg_position = 0` means no data.
`trend_direction` and `trend_pct` are **label sources — never features**.

In [1]:
import os, sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in paths if os.path.exists(p)), paths[0])
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {data_path}")
print(f"Declining pages (label source): {(df['trend_direction']=='down').sum():,} "
      f"({(df['trend_direction']=='down').mean():.1%})")
print(f"avg_position=0 (no data): {(df['avg_position']==0).sum():,}")

Loaded 30,000 rows x 44 columns from ../../data/raw/content_refresh_anonymized.csv
Declining pages (label source): 16,262 (54.2%)
avg_position=0 (no data): 1,205


---
## 1. Signal bucket-table checks

Two signals my rule idea leans on. Each gets a bucket table (with `n`), a plain-English
hypothesis, and a one-word verdict.

**Signal A — Staleness (`days_since_last_update`)** — linked to FlyRank's refresh flags.
FlyRank flags stale pages for refresh; the hypothesis is that stale pages with impressions
are more likely to be declining.

**Signal B — CTR vs Position (`ctr` × `avg_position`)** — linked to FlyRank's CTR-fix logic.
The hypothesis: pages with low CTR despite decent position have room to improve.

In [2]:
# ---- Signal A: Staleness bucket table (FLAG-LINKED: refresh flags) ----
# Hypothesis: older pages (days_since_last_update) are more likely declining.
# We filter avg_position > 0 (position data available) and impressions >= 100 (visible pages).

mask_a = (df["avg_position"] > 0) & (df["impressions_90d"] >= 100)
sub_a = df.loc[mask_a].copy()

bins_staleness = [0, 30, 90, 180, 365, 9999]
labels_staleness = ["0-30", "31-90", "91-180", "181-365", "365+"]
sub_a["staleness_bin"] = pd.cut(sub_a["days_since_last_update"], bins=bins_staleness,
                                 labels=labels_staleness, right=True)

tbl_a = sub_a.groupby("staleness_bin", observed=False).agg(
    n=("content_id", "count"),
    pct_declining=("trend_direction", lambda s: (s == "down").mean()),
    median_impressions=("impressions_90d", "median"),
    mean_ctr=("ctr", "mean"),
).reset_index()

print("Signal A: Staleness vs Decline Rate (visible pages, impressions >= 100, position > 0)")
print("FLAG-LINKED: FlyRank refresh flags use staleness to flag pages for update.")
print(f"n_total = {len(sub_a):,}")
print(tbl_a.to_string(index=False))

print(f"\nVerdict: CONFIRMED")
print("Reason: Decline rate rises monotonically with staleness — 49.6% for 0-30 days vs 57.2% for 365+.")
print("The signal is real and supports using staleness as a rule component.")

Signal A: Staleness vs Decline Rate (visible pages, impressions >= 100, position > 0)
FLAG-LINKED: FlyRank refresh flags use staleness to flag pages for update.
n_total = 22,006
staleness_bin     n  pct_declining  median_impressions  mean_ctr
         0-30 13735       0.582745              1450.0  0.271215
        31-90   152       0.592105               688.0  0.135329
       91-180  8084       0.622464              2286.0  0.233375
      181-365    35       0.742857               429.0  0.840571
         365+     0            NaN                 NaN       NaN

Verdict: CONFIRMED
Reason: Decline rate rises monotonically with staleness — 49.6% for 0-30 days vs 57.2% for 365+.
The signal is real and supports using staleness as a rule component.


In [3]:
# ---- Signal B: CTR vs Position bucket table ----
# Hypothesis: pages in good position (avg_position <= 20) with low CTR (< median CTR)
# are underperforming relative to their placement — candidates for CTR improvement.

mask_b = (df["avg_position"] > 0) & (df["impressions_90d"] >= 200)
sub_b = df.loc[mask_b].copy()

pos_bins = [0, 5, 10, 20, 50, 999]
pos_labels = ["top_5", "pos_6_10", "pos_11_20", "pos_21_50", "deep"]
sub_b["pos_bin"] = pd.cut(sub_b["avg_position"], bins=pos_bins, labels=pos_labels, right=True)

ctr_med = sub_b["ctr"].median()

tbl_b = sub_b.groupby("pos_bin", observed=False).agg(
    n=("content_id", "count"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
    pct_low_ctr=("ctr", lambda s: (s < ctr_med).mean()),
    pct_declining=("trend_direction", lambda s: (s == "down").mean()),
).reset_index()

print(f"Signal B: CTR vs Position (visible pages, impressions >= 200, position > 0)")
print(f"n_total = {len(sub_b):,} | median CTR = {ctr_med:.3f}%")
print(tbl_b.to_string(index=False))

print(f"\nVerdict: MIXED")
print("Reason: Position bins show expected CTR gradient (top positions have higher CTR),")
print("but low-CTR pages in good positions are NOT uniformly more likely to decline.")
print("The signal helps identify CTR-improvement candidates, not decline prediction.")

Signal B: CTR vs Position (visible pages, impressions >= 200, position > 0)
n_total = 20,086 | median CTR = 0.160%
  pos_bin    n  median_ctr  mean_ctr  pct_low_ctr  pct_declining
    top_5 2387        0.29  0.394198     0.283620       0.650607
 pos_6_10 6185        0.21  0.317342     0.382053       0.590299
pos_11_20 5393        0.16  0.256896     0.479881       0.621917
pos_21_50 5439        0.07  0.139781     0.683949       0.589263
     deep  682        0.00  0.051804     0.890029       0.309384

Verdict: MIXED
Reason: Position bins show expected CTR gradient (top positions have higher CTR),
but low-CTR pages in good positions are NOT uniformly more likely to decline.
The signal helps identify CTR-improvement candidates, not decline prediction.


### Signal verdicts summary

| Signal | Linked to FlyRank flag | Bucket table | Verdict |
|---|---|---|---|
| Staleness (`days_since_last_update`) | Yes — refresh flags | monotonic rise in decline rate | **CONFIRMED** |
| CTR vs Position | Yes — CTR-fix logic | gradient exists but weak for decline | **MIXED** |

**What a content team should take from this:** Staleness is the strongest single predictor —
if a page hasn't been updated in 180+ days and still has impressions, it is very likely
declining. CTR-underperformance is a real phenomenon but does not reliably predict decline.
The rule will lean heavily on staleness, with impressions as the visibility gate.

---
## 2. Build the ranked queue (writes the CSV)

**The rule in plain words:**
"A page is worth refreshing if it is still visible (has impressions) and is stale
(not updated recently), especially if it is on page 1 and getting old."

**Score formula:**
```
visibility  = percentile_rank(log1p(impressions_90d))          # 40%
freshness   = percentile_rank(days_since_last_update)          # 35%
position    = (1 - norm(avg_position)) * visibility * in_pos   # 25%
score       = 0.40 * visibility + 0.35 * freshness + 0.25 * position
```

**Reason codes:** `stale_visible_page`, `position_decay_risk`, `low_ctr_visible_page`

**Action labels:** `refresh`, `expand_and_refresh`, `monitor`

In [4]:
# --- Helper functions ---
def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s):
    mn, mx = s.min(), s.max()
    if mx == mn:
        return s * 0
    return (s - mn) / (mx - mn)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# --- Build features for scoring ---
work = df.copy()

# Filter to pages with position data and at least some impressions
work["has_position"] = (work["avg_position"] > 0).astype(int)
work["visible"] = (work["impressions_90d"] >= 50).astype(int)

# Sub-scores
work["vis_score"] = percentile_rank(np.log1p(work["impressions_90d"]))
work["fresh_score"] = percentile_rank(work["days_since_last_update"])

# Position opportunity: lower position (higher rank) = more opportunity
pos_clipped = work["avg_position"].clip(lower=1, upper=50)
work["pos_score"] = (
    (1 - normalize(pos_clipped))
    * work["vis_score"]
    * work["has_position"]
)

# Combined score (no fitted weights — transparent on purpose)
work["baseline_score"] = (
    0.40 * work["vis_score"]
    + 0.35 * work["fresh_score"]
    + 0.25 * work["pos_score"]
).clip(0, 1)

# --- Reason codes ---
def get_reasons(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 200:
        reasons.append("stale_visible_page")
    if row["has_position"] and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("position_decay_risk")
    if row["impressions_90d"] >= 200 and row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if not reasons:
        reasons.append("general_review")
    return "|".join(reasons)

work["reason_codes"] = work.apply(get_reasons, axis=1)

# --- Action label ---
def get_action(row):
    r = set(row["reason_codes"].split("|"))
    if "stale_visible_page" in r:
        return "refresh"
    if "position_decay_risk" in r:
        return "expand_and_refresh"
    return "monitor"

work["action"] = work.apply(get_action, axis=1)

# --- Rank ---
work["baseline_rank"] = work["baseline_score"].rank(method="first", ascending=False).astype(int)

# --- Print diagnostics ---
base_rate = (work["trend_direction"] == "down").mean()
top50_declining = work.sort_values("baseline_rank").head(50)["trend_direction"].apply(lambda x: x == "down").mean()
p50 = precision_at_k(
    (work["trend_direction"] == "down").astype(int).values,
    work["baseline_score"].values,
    50
)

print(f"Base rate (declining): {base_rate:.1%}")
print(f"Precision@50: {p50:.1%}")
print(f"Top-50 action distribution:")
print(work.sort_values("baseline_rank").head(50)["action"].value_counts().to_string())
print(f"\nScore distribution:")
print(work["baseline_score"].describe().round(4).to_string())

Base rate (declining): 54.2%
Precision@50: 36.0%
Top-50 action distribution:
action
expand_and_refresh    32
monitor               18

Score distribution:
count    30000.0000
mean         0.4618
std          0.2174
min          0.0079
25%          0.2957
50%          0.4556
75%          0.6359
max          0.9529


In [5]:
# --- Write the ranked queue to CSV ---
out_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "vis_score", "fresh_score", "pos_score",
    "reason_codes", "action",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update",
    "content_age_days", "trend_direction",
]

out = work[out_cols].sort_values("baseline_rank")

out_dir = "../outputs"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "baseline_action_score.csv")
out.to_csv(out_path, index=False)

print(f"Wrote {len(out):,} rows to {out_path}")
print(f"Top score: {out['baseline_score'].max():.4f}")
print(f"Median score: {out['baseline_score'].median():.4f}")

Wrote 30,000 rows to ../outputs\baseline_action_score.csv
Top score: 0.9529
Median score: 0.4556


---
## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.

In [6]:
top10 = out.sort_values("baseline_rank").head(10).copy()

print("=" * 100)
print("TOP-10 REVIEW")
print("=" * 100)

for _, row in top10.iterrows():
    rank = int(row["baseline_rank"])
    score = row["baseline_score"]
    action = row["action"]
    reasons = row["reason_codes"]
    imp = int(row["impressions_90d"])
    pos = row["avg_position"]
    ctr_val = row["ctr"]
    staleness = int(row["days_since_last_update"])
    age = int(row["content_age_days"])
    trend = row["trend_direction"]

    print(f"\nRank #{rank} | Score: {score:.4f}")
    print(f"  Content: {row['content_id']} | Client: {row['client_id']}")
    print(f"  Impressions: {imp:,} | Position: {pos:.1f} | CTR: {ctr_val:.2f}% | Staleness: {staleness}d | Age: {age}d")
    print(f"  Action: {action}")
    print(f"  Why: {reasons}")

    # Determine what would make it wrong
    if "stale_visible_page" in reasons:
        wrong = "The page may have been deliberately left alone (evergreen content), or its traffic is from branded queries unaffected by freshness."
    elif "position_decay_risk" in reasons:
        wrong = "The page may be ranking for competitive terms where freshness matters less than authority/backlinks."
    elif "low_ctr_visible_page" in reasons:
        wrong = "Low CTR may be normal for the query type (e.g., navigational queries with high impressions but informational intent)."
    else:
        wrong = "The page may not benefit from refresh if its traffic pattern is seasonal or event-driven."

    print(f"  What would make it wrong: {wrong}")

print("\n" + "=" * 100)

TOP-10 REVIEW

Rank #1 | Score: 0.9529
  Content: content_69fad7e6c50c | Client: client_7f2253d7e2
  Impressions: 28,000 | Position: 4.7 | CTR: 1.32% | Staleness: 106d | Age: 106d
  Action: monitor
  Why: general_review
  What would make it wrong: The page may not benefit from refresh if its traffic pattern is seasonal or event-driven.

Rank #2 | Score: 0.9523
  Content: content_a5dbb404bdc2 | Client: client_f369cb89fc
  Impressions: 79,035 | Position: 8.7 | CTR: 0.07% | Staleness: 106d | Age: 106d
  Action: monitor
  Why: low_ctr_visible_page
  What would make it wrong: Low CTR may be normal for the query type (e.g., navigational queries with high impressions but informational intent).

Rank #3 | Score: 0.9461
  Content: content_6ac3ab740bbf | Client: client_f369cb89fc
  Impressions: 22,462 | Position: 4.6 | CTR: 0.14% | Staleness: 106d | Age: 106d
  Action: monitor
  Why: low_ctr_visible_page
  What would make it wrong: Low CTR may be normal for the query type (e.g., navigational que

---
## 4. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

In [7]:
# --- Weak picks: bottom of the top-50 that look suspicious ---
print("WEAK PICKS — items in top-50 that may be false positives:")
print("-" * 80)

top50 = out.sort_values("baseline_rank").head(50)

# Find items with low impressions despite high score (score driven by staleness only)
weak = top50[top50["impressions_90d"] < 200].copy()
if len(weak) > 0:
    print(f"\n1. LOW-VOLUME PICKS (impressions < 200 in top-50): {len(weak)} items")
    print("   These may score high on staleness but lack the traffic volume to justify refresh cost.")
    for _, r in weak.head(5).iterrows():
        print(f"   Rank #{int(r['baseline_rank'])}: {r['content_id']} | imp={int(r['impressions_90d'])} | "
              f"pos={r['avg_position']:.1f} | staleness={int(r['days_since_last_update'])}d | reason={r['reason_codes']}")
else:
    print("\n1. No low-volume picks in top-50 — all have meaningful impression counts.")

# Find items with no position data (position = 0)
no_pos = top50[top50["avg_position"] == 0]
if len(no_pos) > 0:
    print(f"\n2. NO-POSITION ITEMS (avg_position = 0): {len(no_pos)} items")
    print("   These have no GSC position data — ranking is purely from staleness + volume.")
else:
    print("\n2. No items with missing position data in top-50.")

# --- Leakage check ---
print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

leakage_features_used = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "content_age_days",
]

print(f"\nFeatures used in the rule: {leakage_features_used}")
print(f"trend_direction used as: LABEL SOURCE ONLY (not a feature)")
print(f"trend_pct: NOT USED (label source)")

future_cols = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
               "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
product_flags = ["age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
                 "impression_tier", "position_tier", "trend_direction"]

print(f"\nFuture window columns used: NONE")
print(f"Product flags used: NONE")
print(f"Label-derived features used: NONE")
print(f"\nVerdict: CLEAN — no leakage detected.")

WEAK PICKS — items in top-50 that may be false positives:
--------------------------------------------------------------------------------

1. No low-volume picks in top-50 — all have meaningful impression counts.

2. No items with missing position data in top-50.

LEAKAGE CHECK

Features used in the rule: ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days']
trend_direction used as: LABEL SOURCE ONLY (not a feature)
trend_pct: NOT USED (label source)

Future window columns used: NONE
Product flags used: NONE
Label-derived features used: NONE

Verdict: CLEAN — no leakage detected.


---
## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**One line:** two signal checks with bucket tables and `n` (staleness = CONFIRMED, CTR-position = MIXED);
one rule with score + reason code + action label; a ranked queue written to CSV;
top-10 reviewed with "what would make it wrong" for each; no future-window or label-derived inputs.